---
### 1. Necessary imports

In [1]:
import io
import json
import logging
import os
import sys
import uuid
import warnings

import dotenv
from google.adk.agents import LlmAgent
from google.adk.apps.app import App, ResumabilityConfig
from google.adk.events import Event
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.function_tool import FunctionTool
from google.adk.tools.tool_context import ToolContext
from google.genai import types

print("✅ Import success.")

✅ Import success.


---
### 2. Configuration

In [2]:
dotenv.load_dotenv()
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
print("✅ Setup and authentication complete.")

✅ Setup and authentication complete.


---
### 3. Python code runner tool

In [3]:
_safe_globals = {"__builtins__": __builtins__}


def run_python_code(source_code: str, tool_context: ToolContext):
    """
    Run any python code and return the output after execution.
    May requires approval for potentially unsafe codes.
    """

    # When the request requires approval
    if "import" in source_code:
        # Send request confirmation
        if not tool_context.tool_confirmation:
            tool_context.request_confirmation(
                hint="Code marked as potentially unsafe (reason: has import statement).",
            )
            return {
                "status": "pending",
                "message": "Code execution requires approval",
            }

        # User disapprove the request
        elif not tool_context.tool_confirmation.confirmed:
            return {"status": "rejected", "message": "User refused to execute code"}

    # Request doesn't require approval, or is approved by user

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(source_code, _safe_globals)
    finally:
        sys.stdout = old_stdout

    output = buffer.getvalue()
    return {
        "status": "success",
        "output": output,
        "message": "Code executed successfully",
    }


print("✅ Tool created: run_python_code.")

✅ Tool created: run_python_code.


---
### 4: Creating the Agent, App and Runner

#### 4.1: Create the Agent

In [4]:
code_agent = LlmAgent(
    name="code_agent",
    model=Gemini(
        model="gemini-gemini-2.5-flash-lite",
        retry_options=types.HttpRetryOptions(
            attempts=5,  # Maximum retry attempts
            exp_base=7,  # Delay multiplier
            initial_delay=1,
            http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
        ),
    ),
    instruction="""You are a code generator and runner.
Your main tasks involve generating Python codes based on user query, and run it when prompted.
To run code, pass the generated source code as an argument to the run_python_code tool, then send back the output to the user.
After receiving the final result, provide a clear summary including the generated code and the output (if any).
    """,
    tools=[FunctionTool(func=run_python_code)],
)

print("✅ Agent created!")

✅ Agent created!


#### 4.2: Create the Resumable App

In [5]:
app = App(
    name="coordinator",
    root_agent=code_agent,
    resumability_config=ResumabilityConfig(is_resumable=True),
)
print("✅ App created!")

✅ App created!


#### 4.3: Create the Runner

In [6]:
session_service = InMemorySessionService()

runner = Runner(
    app=app,
    session_service=session_service,
)

print("✅ Runner created!")

✅ Runner created!


---
### 5: Run the workflow

#### 5.1: Helper functions

In [7]:
def pprint(*args):
    for arg in args:
        if isinstance(arg, dict):
            print(json.dumps(arg, indent=2))
        else:
            print(arg)
    print()

In [8]:
def create_approval_response(approval_id: str, approved: bool):
    # NOTE: the whole procedure can be describe by the following flow:
    #       tool_context.request_confirmation
    #       -> adk_request_confirmation function called (FunctionCall event)
    #       -> response to adk_request_confirmation (FunctionResponse event)
    #       -> rerun the tool with confirmed = True or False
    #
    #       ⚠️ the following condition MUST be satisfied: ⚠️
    #           response.id == request.function_call.id

    confirmation_response = types.FunctionResponse(
        id=approval_id,
        name="adk_request_confirmation",
        response={"confirmed": approved},
    )
    return types.Content(
        role="user", parts=[types.Part(function_response=confirmation_response)]
    )

In [11]:
def handle_adk_event(event: Event):
    """Handle the event streamed by the Runner.

    If there are adk_request_confirmation
    requests, return a list of their ids.
    """

    # If the event has no content
    if not event.content or not event.content.parts:
        return []

    approval_ids = []

    for part in event.content.parts:
        if part.text:
            pprint("🤖 Agent:", part.text.strip())
        elif part.function_call:
            pprint(
                f"⚙️ Function call ({part.function_call.name}):", part.function_call.args
            )
            if part.function_call.name == "adk_request_confirmation":
                lrti = event.long_running_tool_ids
                assert lrti and part.function_call.id in lrti
                approval_ids.append(part.function_call.id)
        elif part.function_response:
            pprint(
                f"📬 Function resp ({part.function_response.name}):",
                part.function_response.response,
            )

    return approval_ids

#### 5.2: Define the workflow

In [9]:
async def run_workflow(query: str, auto_approve: bool = True):
    pprint("🧑 User:", query)

    user_id = "test_user"
    session_id = uuid.uuid4().hex[:12]
    await session_service.create_session(
        app_name="coordinator", user_id=user_id, session_id=session_id
    )

    approval_ids = []
    invocation_id = None
    query_content = types.Content(role="user", parts=[types.Part(text=query)])

    async for event in runner.run_async(
        user_id=user_id, session_id=session_id, new_message=query_content
    ):
        if invocation_id is None:
            invocation_id = event.invocation_id
            pprint("ℹ️ Invocation ID:", invocation_id)
        approval_ids.extend(handle_adk_event(event))

    # Check if there are any requests waiting for approval
    for approval_id in approval_ids:
        pprint("❗ This adk_request_confirmation ID requires approval!", approval_id)
        print(
            "🤔 Human Decision:", "APPROVED ✅" if auto_approve else "REJECTED ❌", "\n"
        )

        approval_response = create_approval_response(approval_id, auto_approve)

        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            invocation_id=invocation_id,
            new_message=approval_response,
        ):
            # this omits further adk_request_confirmation requests!
            handle_adk_event(event)


print("✅ Created run_workflow")

✅ Created run_workflow


In [12]:
await run_workflow(
    "write a function to calculate sum of all "
    "digits of a number. then test it with: 123123"
)

🧑 User:
write a function to calculate sum of all digits of a number. then test it with: 123123

ℹ️ Invocation ID:
e-4ade9eca-e3b9-473b-96d1-6918e5450f97

⚙️ Function call (run_python_code):
{
  "source_code": "\ndef sum_digits(n):\n    s = 0\n    while n > 0:\n        s += n % 10\n        n //= 10\n    return s\n\nprint(sum_digits(123123))\n"
}

📬 Function resp (run_python_code):
{
  "status": "success",
  "output": "12\n",
  "message": "Code executed successfully"
}

🤖 Agent:
The sum of the digits of 123123 is 12. The Python code to achieve this is:

🤖 Agent:
```python
def sum_digits(n):
    s = 0
    while n > 0:
        s += n % 10
        n //= 10
    return s

print(sum_digits(123123))
```



In [13]:
await run_workflow(
    "write and run a python code that prints the square root of 2", auto_approve=True
)

🧑 User:
write and run a python code that prints the square root of 2

ℹ️ Invocation ID:
e-2cabd859-bb07-40e2-b351-10ed924f9bc6

⚙️ Function call (run_python_code):
{
  "source_code": "import math\nprint(math.sqrt(2))"
}

⚙️ Function call (adk_request_confirmation):
{
  "originalFunctionCall": {
    "id": "adk-6416f384-15a8-43a2-8225-514309cd95bc",
    "args": {
      "source_code": "import math\nprint(math.sqrt(2))"
    },
    "name": "run_python_code"
  },
  "toolConfirmation": {
    "hint": "Code marked as potentially unsafe (reason: has import statement).",
    "confirmed": false
  }
}

📬 Function resp (run_python_code):
{
  "status": "pending",
  "message": "Code execution requires approval"
}

❗ This adk_request_confirmation ID requires approval!
adk-19e77fe7-fa9a-4d97-91dc-1e9f4317ff05

🤔 Human Decision: APPROVED ✅ 

📬 Function resp (run_python_code):
{
  "status": "success",
  "output": "1.4142135623730951\n",
  "message": "Code executed successfully"
}

🤖 Agent:
The code calcul

In [14]:
await run_workflow(
    "write and run a python code that prints the square root of 3",
    auto_approve=False,
)

🧑 User:
write and run a python code that prints the square root of 3

ℹ️ Invocation ID:
e-09130472-994f-427d-822a-5105a7b760a8

⚙️ Function call (run_python_code):
{
  "source_code": "import math\nprint(math.sqrt(3))"
}

⚙️ Function call (adk_request_confirmation):
{
  "originalFunctionCall": {
    "id": "adk-a1189d59-e4f7-44c6-9750-c49106c95145",
    "args": {
      "source_code": "import math\nprint(math.sqrt(3))"
    },
    "name": "run_python_code"
  },
  "toolConfirmation": {
    "hint": "Code marked as potentially unsafe (reason: has import statement).",
    "confirmed": false
  }
}

📬 Function resp (run_python_code):
{
  "status": "pending",
  "message": "Code execution requires approval"
}

❗ This adk_request_confirmation ID requires approval!
adk-5d377a58-482f-4277-bf90-183c2508cebc

🤔 Human Decision: REJECTED ❌ 

📬 Function resp (run_python_code):
{
  "status": "rejected",
  "message": "User refused to execute code"
}

